In [25]:
# Import relevant packages
import numpy as np
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import os

### Read in processed files 

In [26]:
# Read in processed data files
df_energy_clean = pd.read_csv('../data/processed/energy_dataset_clean.csv')
df_weather_wide = pd.read_csv('../data/processed/weather_wide.csv')

In [27]:
df_energy_clean.columns

Index(['time', 'date', 'hour', 'generation biomass',
       'generation fossil brown coal/lignite', 'generation fossil gas',
       'generation fossil hard coal', 'generation fossil oil',
       'generation hydro pumped storage consumption',
       'generation hydro run-of-river and poundage',
       'generation hydro water reservoir', 'generation nuclear',
       'generation other', 'generation other renewable', 'generation solar',
       'generation waste', 'generation wind onshore',
       'forecast solar day ahead', 'forecast wind onshore day ahead',
       'total load forecast', 'total load actual', 'price day ahead',
       'price actual'],
      dtype='object')

### Grouping generation sources by type 

#### feel free to change the groupings 

In [28]:
solar_cols        = ['generation solar']
wind_cols         = ['generation wind onshore']
hydro_cols        = ['generation hydro run-of-river and poundage',
                     'generation hydro water reservoir']
pumped_hydro_cols = ['generation hydro pumped storage consumption']
fossil_cols       = ['generation fossil brown coal/lignite',
                     'generation fossil gas',
                     'generation fossil hard coal',
                     'generation fossil oil']
nuclear_cols      = ['generation nuclear']
other_cols        = ['generation other', 'generation other renewable', 
                     'generation biomass']  # biomass added
# 'generation waste' dropped entirely

df = df_energy_clean.copy()

df['gen_solar']        = df[solar_cols].sum(axis=1)
df['gen_wind']         = df[wind_cols].sum(axis=1)
df['gen_hydro']        = df[hydro_cols].sum(axis=1)
df['gen_pumped_hydro'] = df[pumped_hydro_cols].sum(axis=1)
df['gen_fossil']       = df[fossil_cols].sum(axis=1)
df['gen_nuclear']      = df[nuclear_cols].sum(axis=1)
df['gen_other']        = df[other_cols].sum(axis=1)

gen_cols_to_drop = (solar_cols + wind_cols + hydro_cols + pumped_hydro_cols +
                    fossil_cols + nuclear_cols + other_cols +
                    ['generation waste'])  # dropped but not summed
df.drop(columns=gen_cols_to_drop, inplace=True)

df_energy_clean = df.copy()

In [29]:
df_energy_clean

,time,date,hour,forecast solar day ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual,gen_solar,gen_wind,gen_hydro,gen_pumped_hydro,gen_fossil,gen_nuclear,gen_other
0,2014-12-31 23:00:00+00:00,2014-12-31,23,17.0,6436.0,26118.0,25385.0,50.10,65.41,49.0,6378.0,2950.0,863.0,10156.0,7096.0,563.0
1,2015-01-01 00:00:00+00:00,2015-01-01,0,16.0,5856.0,24934.0,24382.0,48.10,64.92,50.0,5890.0,2667.0,920.0,10437.0,7096.0,563.0
2,2015-01-01 01:00:00+00:00,2015-01-01,1,8.0,5454.0,23515.0,22734.0,47.33,64.48,50.0,5461.0,2344.0,1164.0,9918.0,7099.0,564.0
3,2015-01-01 02:00:00+00:00,2015-01-01,2,2.0,5151.0,22642.0,21286.0,42.27,59.32,50.0,5238.0,1728.0,1503.0,8859.0,7098.0,556.0
4,2015-01-01 03:00:00+00:00,2015-01-01,3,9.0,4861.0,21785.0,20264.0,38.41,56.04,42.0,4935.0,1673.0,1826.0,8313.0,7097.0,545.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35059,2018-12-31 18:00:00+00:00,2018-12-31,18,96.0,3253.0,30619.0,30653.0,68.85,77.02,85.0,3113.0,5971.0,1.0,10440.0,6073.0,455.0
35060,2018-12-31 19:00:00+00:00,2018-12-31,19,51.0,3353.0,29932.0,29735.0,68.40,76.16,33.0,3288.0,5103.0,1.0,9981.0,6074.0,453.0
35061,2018-12-31 20:00:00+00:00,2018-12-31,20,36.0,3404.0,27903.0,28071.0,66.88,74.30,31.0,3503.0,3979.0,50.0,9615.0,6076.0,447.0
35062,2018-12-31 21:00:00+00:00,2018-12-31,21,29.0,3273.0,25450.0,25801.0,63.93,69.89,31.0,3586.0,3196.0,108.0,9018.0,6075.0,447.0


### Merge weather & energy dfs 

In [30]:
df_energy_weather = df_energy_clean.merge(
    df_weather_wide.drop(columns=["date", "hour"]),
    on="time",
    how="inner",
    validate="one_to_one"
)

In [31]:
df_energy_weather

,time,date,hour,forecast solar day ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual,gen_solar,...,snow_3h_Barcelona,snow_3h_Bilbao,snow_3h_Madrid,snow_3h_Seville,snow_3h_Valencia,clouds_all_Barcelona,clouds_all_Bilbao,clouds_all_Madrid,clouds_all_Seville,clouds_all_Valencia
0,2014-12-31 23:00:00+00:00,2014-12-31,23,17.0,6436.0,26118.0,25385.0,50.10,65.41,49.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
1,2015-01-01 00:00:00+00:00,2015-01-01,0,16.0,5856.0,24934.0,24382.0,48.10,64.92,50.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
2,2015-01-01 01:00:00+00:00,2015-01-01,1,8.0,5454.0,23515.0,22734.0,47.33,64.48,50.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
3,2015-01-01 02:00:00+00:00,2015-01-01,2,2.0,5151.0,22642.0,21286.0,42.27,59.32,50.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
4,2015-01-01 03:00:00+00:00,2015-01-01,3,9.0,4861.0,21785.0,20264.0,38.41,56.04,42.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35059,2018-12-31 18:00:00+00:00,2018-12-31,18,96.0,3253.0,30619.0,30653.0,68.85,77.02,85.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
35060,2018-12-31 19:00:00+00:00,2018-12-31,19,51.0,3353.0,29932.0,29735.0,68.40,76.16,33.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
35061,2018-12-31 20:00:00+00:00,2018-12-31,20,36.0,3404.0,27903.0,28071.0,66.88,74.30,31.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0
35062,2018-12-31 21:00:00+00:00,2018-12-31,21,29.0,3273.0,25450.0,25801.0,63.93,69.89,31.0,...,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0


In [32]:
df_energy_weather.columns

Index(['time', 'date', 'hour', 'forecast solar day ahead',
       'forecast wind onshore day ahead', 'total load forecast',
       'total load actual', 'price day ahead', 'price actual', 'gen_solar',
       'gen_wind', 'gen_hydro', 'gen_pumped_hydro', 'gen_fossil',
       'gen_nuclear', 'gen_other', 'temp_Barcelona', 'temp_Bilbao',
       'temp_Madrid', 'temp_Seville', 'temp_Valencia', 'pressure_Barcelona',
       'pressure_Bilbao', 'pressure_Madrid', 'pressure_Seville',
       'pressure_Valencia', 'humidity_Barcelona', 'humidity_Bilbao',
       'humidity_Madrid', 'humidity_Seville', 'humidity_Valencia',
       'wind_speed_Barcelona', 'wind_speed_Bilbao', 'wind_speed_Madrid',
       'wind_speed_Seville', 'wind_speed_Valencia', 'wind_deg_Barcelona',
       'wind_deg_Bilbao', 'wind_deg_Madrid', 'wind_deg_Seville',
       'wind_deg_Valencia', 'rain_1h_Barcelona', 'rain_1h_Bilbao',
       'rain_1h_Madrid', 'rain_1h_Seville', 'rain_1h_Valencia',
       'rain_3h_Barcelona', 'rain_3h_Bilbao',